# Installations

In [ ]:
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git

In [ ]:
!pip install insightface onnxruntime-gpu --quiet

In [ ]:
!pip install kagglehub

In [ ]:
!pip install facenet-pytorch ninja lpips torchmetrics --force-reinstall --no-cache-dir

# Imports

In [ ]:
import sys
sys.path.insert(0, "/content/stylegan2-ada-pytorch")

In [ ]:
import os

# Create a folder for models
os.makedirs('models', exist_ok=True)

# Download the stylegan2-ada-pytorch FFHQ model (resolution 1024x1024)
# This is hosted by NVIDIA
!wget https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl -O models/ffhq.pkl

print("Download complete.")

In [ ]:
import torch
import pickle
import copy
import dnnlib
import legacy # From the cloned repo

import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
from facenet_pytorch import InceptionResnetV1
from torchvision import transforms
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.distance import cosine
from tqdm import tqdm
from torchmetrics.image import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio
import lpips
import math
import random
import shutil
import kagglehub
import insightface
from insightface.app import FaceAnalysis

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

# Models Definition

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Ensure deterministic behavior (might slow down slightly)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(0)

In [ ]:
class StyleGANGenerator(torch.nn.Module):
    def __init__(self, network_pkl):
        super(StyleGANGenerator, self).__init__()
        print(f'Loading network from "{network_pkl}"...')

        with dnnlib.util.open_url(network_pkl) as f:
            # Load the network from the pickle file
            network_dict = legacy.load_network_pkl(f)
            self.G = network_dict['G_ema'].to(device)
            self.D = network_dict['D'].to(device)
            self.D.eval()
            for param in self.D.parameters():
                param.requires_grad = False

        # Lock the weights (we never train the generator itself)
        self.G.eval()
        for param in self.G.parameters():
            param.requires_grad = False

        # Store useful constants
        self.w_dim = self.G.w_dim  # Usually 512
        self.num_ws = self.G.mapping.num_ws # Usually 18 for 1024x1024
        print(f'Loaded network! (w_dim: {self.w_dim}, num_ws: {self.num_ws})')

    def forward(self, w_plus_vector):
        """
        Input: w_plus_vector of shape (Batch, 18, 512)
        Output: Image tensor (Batch, 3, 1024, 1024) in range [-1, 1]
        """
        # synthesis() expects input to be split by layers, but w+ is already shaped correctly
        # noise_mode='const' means we don't add random noise to hair/pores every time (deterministic)
        img = self.G.synthesis(w_plus_vector, noise_mode='const')
        return img

    def get_mean_w(self, n_samples=4096, seed=0):
        """
        Get the average latent code (W space).
        Optimizing starting from the Mean Face is much faster/easier.
        """
        torch.manual_seed(seed)
        z = torch.randn(n_samples, self.G.z_dim, device=device)
        w = self.G.mapping(z, None) # Convert z to w
        w_avg = w.mean(0, keepdim=True)

        return w_avg

# Initialize the model
generator = StyleGANGenerator('models/ffhq.pkl')
print("Generator Loaded Successfully!")

In [ ]:
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
transform = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor()
])

# Function Definitions

### Display Result function

In [ ]:
def save_and_display_image(image_tensor, filename=None, normalize_for_viewing=False, caption=""):
    """
    normalize_for_viewing:
       If True: Stretches min/max to 0-1 (Best for seeing the NOISE PATTERN).
       If False: Clips to valid 0-1 range (Best for seeing VALID PIXELS).
    """
    img = image_tensor.detach().cpu().squeeze(0)

    if normalize_for_viewing:
        # Map min->0, max->1
        # Good for Gaussian
        img = (img - img.min()) / (img.max() - img.min())
    else:
        # Clip to valid range
        img = (torch.clamp(img, -1, 1) + 1) / 2.0

    final_image_pil = transforms.ToPILImage()(img)
    if filename:
        final_image_pil.save(filename)

    img_np = img.permute(1, 2, 0).numpy()

    plt.figure(figsize=(4, 4))
    plt.imshow(img_np)
    plt.axis('off')
    plt.title(caption, y=-0.08, fontsize=12)

    plt.show()

### Metric Functions

In [ ]:
lpips_metric = lpips.LPIPS(net='vgg').to(device)
# < 0.25 high similarity
# > 0.7 different images

psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
# > 30 dB: High quality (hard to distinguish difference).
# 20-30 dB: Acceptable quality.
# < 20 dB: Poor quality (very noisy).

ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
# 1.0: Identical images.
# > 0.9: Very structurally similar.

In [ ]:
class ArcFaceMetric:
    def __init__(self, device='cuda'):
        # Load the default 'buffalo_l' model pack (contains ResNet50 ArcFace)
        self.app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])
        self.app.prepare(ctx_id=0, det_size=(640, 640))
        self.handler = self.app.models['recognition'] # The ArcFace recognition model
        self.device = device

    def get_embedding(self, tensor_img):
        """
        Input: Tensor [1, 3, H, W] in range [-1, 1] or [0, 1], RGB
        Output: Numpy Array [512]
        """
        # 1. Convert Tensor to Numpy Image [H, W, 3] in range [0, 255]
        # Assuming input is [-1, 1] (tanh output)
        if tensor_img.min() < 0:
            img = (tensor_img * 0.5 + 0.5)
        else:
            img = tensor_img

        img = img.clamp(0, 1).cpu().detach().squeeze(0).permute(1, 2, 0).numpy()
        img = (img * 255).astype(np.uint8)

        # 2. Convert RGB to BGR (InsightFace expects BGR via OpenCV)
        img_bgr = img[:, :, ::-1]

        # 3. Resize to 112x112 (ArcFace standard input)
        # Note: Usually we use alignment (warping), but for simple metric
        # on already cropped faces, resizing is acceptable.
        import cv2
        img_resized = cv2.resize(img_bgr, (112, 112))

        # 4. Get Embedding (blob is a formatting helper)
        blob = cv2.dnn.blobFromImage(img_resized, 1.0 / 127.5, (112, 112), (127.5, 127.5, 127.5), swapRB=True)
        # Note: InsightFace handler expects raw forward pass usually,
        # but calling .get_feat is safer if wrapping 'get'
        # We can simulate the forward pass directly on the handler:
        embedding = self.handler.get_feat(img_resized)

        return embedding.flatten()

    def compute_sim(self, tensor_gen, tensor_target):
        emb_gen = self.get_embedding(tensor_gen)
        emb_target = self.get_embedding(tensor_target)

        # Compute Cosine Similarity
        from numpy.linalg import norm
        sim = np.dot(emb_gen, emb_target) / (norm(emb_gen) * norm(emb_target))
        return sim

# Initialize ONCE (it takes time to load)
arcface_metric = ArcFaceMetric(device=device)

In [ ]:
def evaluate_and_log(i, iterations, current_img, target_img, history, freq=20, grayscale=False):
    """
    Evaluates metrics and updates history lists in-place.

    Args:
        i (int): Current iteration.
        iterations (int): Total iterations.
        current_img (Tensor): The normalized image (output of tanh, [-1, 1]).
        target_img (Tensor): The target image ([0, 1]).
        history (tuple): (lpips_list, psnr_list, ssim_list, steps).
        freq (int): Log frequency.
    """
    if i % freq != 0 and i != iterations - 1:
        return

    lpips_list, psnr_list, ssim_list, id_score_list, steps = history

    with torch.no_grad():
        # Convert [-1, 1] -> [0, 1]
        val_img = (current_img * 0.5) + 0.5
        tgt_img = target_img

        # Clamp to ensure numerical stability (fix float errors like -0.0001 or 1.0001)
        val_img = val_img.clamp(0, 1)
        tgt_img = tgt_img.clamp(0, 1)

        if grayscale:
            val_img = transforms.functional.rgb_to_grayscale(val_img, num_output_channels=3)
            tgt_img = transforms.functional.rgb_to_grayscale(tgt_img, num_output_channels=3)

        lpips_list.append(lpips_metric(val_img, tgt_img).item())
        psnr_list.append(psnr_metric(val_img, tgt_img).item())
        ssim_list.append(ssim_metric(val_img, tgt_img).item())

        arcface_val_img = val_img * 2 - 1
        arcface_tgt_img = tgt_img * 2 - 1
        id_sim = arcface_metric.compute_sim(arcface_val_img, arcface_tgt_img)
        id_score_list.append(id_sim)

        steps.append(i)

### Data Loading

In [ ]:
path = kagglehub.dataset_download("badasstechie/celebahq-resized-256x256")
print("Path to dataset files:", path)

In [ ]:
def load_image(path: str):
    """ Load the image from its path, and calculate its embedding """
    img = Image.open(path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)
    emb = model(x*2-1).detach().cpu()
    emp = emb.to(device)
    return emb, x

### Main Attack Function

In [ ]:
def inversion_attack(target_embedding, target_image, seed=0, iterations_on_w=25, iterations_on_w_plus=35, evaluate=True):
    # Initialization
    w_avg = generator.get_mean_w(seed=seed)
    w_single = w_avg[:, 0, :].clone()
    w_single.requires_grad = True

    optimizer_w_single = optim.Adam([w_single], lr=0.05)
    scheduler_w_single = torch.optim.lr_scheduler.LinearLR(optimizer_w_single, start_factor=1.0, end_factor=0.5, total_iters=iterations_on_w)
    mse_loss = torch.nn.MSELoss()

    loss_list = []
    cosine_similarity_list = []

    lpips_list = []
    psnr_list = []
    ssim_list = []
    arcface_score_list = []
    steps = []
    evaluation_metrics = (lpips_list, psnr_list, ssim_list, arcface_score_list, steps)

    # Optimization on W:
    for i in tqdm(range(iterations_on_w)):
        optimizer_w_single.zero_grad()

        # Copy w 18 times
        w_stack = w_single.unsqueeze(1).repeat(1, 18, 1)

        # Generate the image with StyleGAN
        generated_image_1024 = generator(w_stack)

        # Resize for FaceNet (160x160)
        generated_image_160 = F.interpolate(generated_image_1024, size=(160, 160), mode='bilinear', align_corners=False)

        # Get FaceNet embedding
        current_embedding = model(generated_image_160)

        # Calculate loss
        loss = mse_loss(current_embedding, target_embedding)

        cos_sim = nn.functional.cosine_similarity(current_embedding, target_embedding).item()
        evaluate_and_log(i, iterations_on_w, generated_image_160, target_image, evaluation_metrics, freq=10)

        loss.backward()
        optimizer_w_single.step()
        scheduler_w_single.step()

        if i == 0 or (i + 1) % 100 == 0:
            print(f"Step [{i+1}/{iterations_on_w}], Loss: {loss.item():.6f}")

        loss_list.append(loss.item())
        cosine_similarity_list.append(cos_sim)

    # Get final w
    w_plus = w_single.unsqueeze(1).repeat(1, 18, 1).detach().clone()
    w_plus.requires_grad = True
    optimizer_w_plus = optim.Adam([w_plus], lr=0.025)
    scheduler_w_plus = torch.optim.lr_scheduler.LinearLR(optimizer_w_plus, start_factor=1.0, end_factor=0.25, total_iters=iterations_on_w_plus)

    for i in tqdm(range(iterations_on_w_plus)):
        optimizer_w_plus.zero_grad()

        # Generate the image with StyleGAN
        generated_image_1024 = generator(w_plus)

        # Resize for FaceNet (160x160)
        generated_image_160 = F.interpolate(generated_image_1024, size=(160, 160), mode='bilinear', align_corners=False)

        # Get FaceNet embedding
        current_embedding = model(generated_image_160)

        # Calculate loss
        loss = mse_loss(current_embedding, target_embedding)

        cos_sim = nn.functional.cosine_similarity(current_embedding, target_embedding).item()
        evaluate_and_log(i + iterations_on_w, iterations_on_w_plus + iterations_on_w, generated_image_160, target_image, evaluation_metrics, freq=10)

        loss.backward()
        optimizer_w_plus.step()
        scheduler_w_plus.step()

        if i == 0 or (i + 1) % 100 == 0:
            print(f"Step [{i+1}/{iterations_on_w_plus}], Loss: {loss.item():.6f}")

        loss_list.append(loss.item())
        cosine_similarity_list.append(cos_sim)

    final_image = generated_image_1024.detach().cpu().squeeze(0)
    print(f"Inversion Complete.")
    return  final_image, arcface_score_list[-1]

# Run Attack

In [ ]:
image_path = f"{path}/celeba_hq_256/00002.jpg"
seed = 1
target_embedding, target_image = load_image(image_path)

with torch.no_grad():
    save_and_display_image(target_image, normalize_for_viewing=True, caption=f"Target Image")

In [ ]:
final_image, arcface_score = inversion_attack(target_embedding.to(device), target_image.to(device), seed=seed)

In [ ]:
with torch.no_grad():
    save_and_display_image(final_image, caption=f"Reconstructed Image. ArcFace: {arcface_score:.3f}")
    save_and_display_image(target_image, normalize_for_viewing=True, caption=f"Target Image")